# Tahap 1: Pre-processing & Ekstraksi ROI Wajah dari Video

**Judul Skripsi:** Deteksi Mikro Ekspresi Wajah pada Rekaman Video Menggunakan Arsitektur Graph Convolutional Network (GCN)

---

## Latar Belakang Pipeline

Sistem deteksi mikro ekspresi pada penelitian ini dirancang secara **dua tahap (two-stage pipeline)**:

| Tahap | Fungsi | Output |
|-------|--------|--------|
| **Tahap 1 (Notebook ini)** | Pre-processing video → deteksi wajah (YOLOv5) → cropping ROI wajah per frame | Urutan gambar wajah (`frame_XXXX.jpg`) |
| **Tahap 2 (Terpisah)** | Klasifikasi spatio-temporal (3D-CNN / ViT / **GCN**) | Label mikro ekspresi |

> **Catatan akademis:** Notebook ini **TIDAK** melakukan pelatihan model klasifikasi ekspresi. Fokusnya murni pada ekstraksi Region of Interest (ROI) wajah sebagai persiapan input untuk model GCN pada tahap berikutnya.

**Dataset video contoh:** CASME II, SAMM, atau video uji di folder `input_videos/`.

In [1]:
import sys
print(sys.executable)

/opt/anaconda3/envs/microsense/bin/python


In [2]:
import sys
!{sys.executable} -m pip install --force-reinstall numpy==1.26.4 scipy==1.11.4

ERROR: Could not find a version that satisfies the requirement numpy==1.26.4 (from versions: none)
ERROR: No matching distribution found for numpy==1.26.4


## 1. Setup & Import Library

In [3]:
!pip install pandas
!pip install requests
!pip install PyYAML
!pip install seaborn matplotlib pillow
!pip install thop

In [4]:
# =============================================================================
# Import library + validasi environment (wajib sebelum load model)
# =============================================================================
import os
import sys
from pathlib import Path

import numpy as np
import cv2
import pandas as pd
import torch
from tqdm.auto import tqdm

# --- Cek kompatibilitas NumPy (torch 2.1.x butuh numpy 1.x) ---
_np_major = int(np.__version__.split('.')[0])
if _np_major >= 2:
    raise RuntimeError(
        f'NumPy {_np_major}.x terdeteksi ({np.__version__}) — TIDAK kompatibel dengan torch 2.1.x.\n'
        'Solusi (jalankan di terminal, lalu Restart Kernel):\n'
        '  conda activate microsense\n'
        '  pip install "numpy>=1.23.5,<2.0" "scipy>=1.10.0" opencv-python --force-reinstall'
    )

# Konfigurasi perangkat komputasi
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

# YOLOv5-Face WAJIB inferensi di CPU pada Apple Silicon.
# PyTorch MPS menghasilkan bbox/landmark bergeser ke kiri-atas (bug numerik backend MPS).
YOLO_DEVICE = 'cpu' if DEVICE == 'mps' else DEVICE

print(f'NumPy          : {np.__version__}')
print(f'OpenCV         : {cv2.__version__}')
print(f'PyTorch        : {torch.__version__}')
print(f'Perangkat aktif: {DEVICE}')
print(f'YOLO inferensi : {YOLO_DEVICE}  (face detection)')

NumPy          : 1.26.4
OpenCV         : 4.11.0
PyTorch        : 2.13.0
Perangkat aktif: mps
YOLO inferensi : cpu  (face detection)


/opt/anaconda3/envs/microsense/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Konfigurasi Path & Hyperparameter Pre-processing

In [5]:
# =============================================================================
# Path direktori — semua relatif terhadap folder python test (lokasi notebook)
# =============================================================================
NOTEBOOK_DIR = Path('.').resolve()
PROJECT_ROOT = NOTEBOOK_DIR

INPUT_VIDEOS_DIR = NOTEBOOK_DIR / 'input_videos'
OUTPUT_CROPPED_FACES_DIR = NOTEBOOK_DIR / 'output_cropped_faces'

# Repo arsitektur YOLOv5-Face lokal (clone deepcam-cn/yolov5-face)
YOLOV5_FACE_REPO = NOTEBOOK_DIR / 'yolov5-face'
FACE_WEIGHTS_PATH = NOTEBOOK_DIR / 'yolov5m-face.pt'

VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.MP4', '.AVI', '.MOV'}

# --- Hyperparameter deteksi & cropping ---
CONF_THRESHOLD = 0.50            # ambang confidence YOLOv5-Face
FACE_CLASS_ID = 0                # kelas 'face' pada yolov5m-face.pt
CROP_SIZE = (224, 224)
FRAME_STRIDE = 1

MIN_FACE_HEIGHT_PX = 20          # abaikan bbox wajah terlalu kecil (data sampah)
INFERENCE_SIZE = 640             # resolusi letterbox YOLOv5-Face (akan di-snap ke stride model)
BBOX_PADDING_RATIO = 0.15        # margin 15% di sekitar bbox (dahi, alis, dagu, bibir)
SQUARE_CROP = True               # crop persegi agar proporsi wajah konsisten untuk GCN
FOREHEAD_BIAS_RATIO = 0.06       # geser pusat crop sedikit ke atas (lebih banyak dahi)
ENABLE_FACE_ALIGNMENT = True     # rotasi crop wajah agar mata sejajar horizontal (landmark-based)

# Filter heuristik wajah — rasio tinggi/lebar (h/w)
FACE_ASPECT_RATIO_MIN = 0.75     # toleransi bbox agak melebar (floating-point / pose miring)
FACE_ASPECT_RATIO_MAX = 1.8      # tolak bbox terlalu lonjong

# Validasi kualitas crop sebelum resize
MIN_CROP_STD = 5.0               # std dev terlalu rendah → gambar flat/noise
MIN_CROP_MEAN = 10.0             # terlalu gelap → kemungkinan crop kosong
BLACK_PIXEL_THRESHOLD = 15
MAX_BLACK_PIXEL_RATIO = 0.95     # >95% piksel hitam → buang frame

INPUT_VIDEOS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CROPPED_FACES_DIR.mkdir(parents=True, exist_ok=True)

print('Input videos :', INPUT_VIDEOS_DIR)
print('Output faces :', OUTPUT_CROPPED_FACES_DIR)
print('Model weights:', FACE_WEIGHTS_PATH)

Input videos : /Users/stefanieagahari/Micro-Expression-Detector/python test/input_videos
Output faces : /Users/stefanieagahari/Micro-Expression-Detector/python test/output_cropped_faces
Model weights: /Users/stefanieagahari/Micro-Expression-Detector/python test/yolov5m-face.pt


## 3. Inisialisasi Model Deteksi Wajah (YOLOv5-Face — Local Source)

### Latar Belakang Akademis

Modul deteksi wajah pada tahap pre-processing ini diadaptasi dari repositori open-source **[deepcam-cn/yolov5-face](https://github.com/deepcam-cn/yolov5-face)**, yang merupakan fork arsitektur YOLOv5 khusus untuk *face detection*. Repositori tersebut dikloning ke folder lokal `yolov5-face/` di dalam `python test/`, sehingga inferensi dapat dilakukan **offline** tanpa mengunduh model dari internet.

YOLOv5 standar (dilatih pada dataset COCO) hanya mendeteksi kelas `person` (tubuh manusia), sehingga sering salah memotong area bahu atau pakaian. Oleh karena itu, penelitian ini mengimplementasikan arsitektur **YOLO5Face** (berdasarkan repositori deepcam-cn/yolov5-face). Model ini dimodifikasi dengan **5-point landmark regression head** dan dilatih khusus menggunakan dataset **WiderFace**, sehingga menjamin lokalisasi wajah yang sangat presisi bahkan pada kondisi *low-light* atau resolusi kecil.

### Spesifikasi Inferensi

| Parameter | Nilai | Keterangan |
|-----------|-------|------------|
| Sumber arsitektur | `yolov5-face/` (local) | Clone repo deepcam-cn/yolov5-face |
| Weights | `yolov5m-face.pt` | Pre-trained face detector (medium) |
| Confidence threshold | `0.50` | Lebih tinggi dari COCO karena model face-specific |
| Kelas target | `face` (id=0) | Bukan kelas `person` COCO |
| Perangkat inferensi | **CPU** (Mac MPS) | Wajib CPU di Apple Silicon — MPS menghasilkan bbox/landmark salah |

In [ ]:
# =============================================================================
# INISIALISASI MODEL — YOLOv5-Face (repo lokal + yolov5m-face.pt)
# =============================================================================

def load_face_detection_model():
    """
    Memuat model YOLOv5-Face dari repo lokal yolov5-face/ dan weights yolov5m-face.pt.
    Arsitektur dibaca dari folder lokal — tidak mengunduh dari internet.
    """
    if not YOLOV5_FACE_REPO.is_dir():
        raise FileNotFoundError(
            f'Repo YOLOv5-Face tidak ditemukan: {YOLOV5_FACE_REPO}\n'
            'Clone repositori ke python test/: '
            'git clone https://github.com/deepcam-cn/yolov5-face yolov5-face'
        )

    if not FACE_WEIGHTS_PATH.exists():
        raise FileNotFoundError(
            f'Weights tidak ditemukan: {FACE_WEIGHTS_PATH}\n'
            'Pastikan file yolov5m-face.pt ada di folder python test/.'
        )

    print(f'Repo arsitektur : {YOLOV5_FACE_REPO.resolve()}')
    print(f'File weights    : {FACE_WEIGHTS_PATH.resolve()}')

    repo_path = str(YOLOV5_FACE_REPO.resolve())
    weights_path = str(FACE_WEIGHTS_PATH.resolve())

    # Bersihkan cache import — cegah bentrok dengan paket `utils`/`models` lain di sesi Jupyter
    for mod_name in list(sys.modules):
        if mod_name in {'utils', 'models', 'hubconf'} or mod_name.startswith(('utils.', 'models.')):
            del sys.modules[mod_name]

    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    # Muat yolov5m-face.pt (logika hubconf.custom) + fuse Conv+BN seperti detect_face.py
    # Tidak memakai autoshape — pipeline inferensi eksplisit di detect_faces_in_frame().
    ckpt = torch.load(weights_path, map_location='cpu', weights_only=False)
    pretrained = ckpt.get('model', ckpt) if isinstance(ckpt, dict) else ckpt
    pretrained = pretrained.float()

    yolo_core = pretrained.fuse().eval().to(YOLO_DEVICE)
    yolo_core.names = getattr(pretrained, 'names', {0: 'face'})

    class YOLOFaceWrapper:
        """Wrapper tipis agar konfigurasi conf/classes konsisten di seluruh notebook."""

        def __init__(self, core, names):
            self.model = core
            self.names = names
            self.conf = CONF_THRESHOLD
            self.iou = 0.45
            self.classes = [FACE_CLASS_ID]

        def eval(self):
            self.model.eval()
            return self

    model = YOLOFaceWrapper(yolo_core, yolo_core.names)
    model.eval()
    return model


yolo_model = load_face_detection_model()
active_classes = yolo_model.classes if yolo_model.classes is not None else [FACE_CLASS_ID]

print(f'Model siap | YOLOv5m-Face (fused) | conf={yolo_model.conf} | device={YOLO_DEVICE}')
print('Kelas aktif:', [yolo_model.names[int(c)] for c in active_classes])

Repo arsitektur : /Users/stefanieagahari/Micro-Expression-Detector/python test/yolov5-face
File weights    : /Users/stefanieagahari/Micro-Expression-Detector/python test/yolov5m-face.pt
Fusing layers... 
Model siap | YOLOv5m-Face (fused) | conf=0.5 | device=cpu
Kelas aktif: ['face']


/opt/anaconda3/envs/microsense/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4217.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


## 4. Fungsi Ekstraksi Wajah dari Video

Pipeline per frame:
1. **Baca frame mentah** — `cv2.VideoCapture` → letterbox + inferensi YOLOv5-Face + `scale_coords` ke resolusi asli
2. **YOLOv5-Face** — deteksi wajah via model lokal deepcam-cn/yolov5-face (NMS face + landmark head)
3. **Filter Heuristik Aspect Ratio** — rasio h/w antara 0.8–1.8 (tolak bbox bahu/bayangan)
4. **Filter Heuristik Center Bias** — pilih bbox terdekat titik tengah layar
5. **Safe Cropping** — `scale_coords` letterbox→frame asli, padding proporsional, crop persegi + clamp simetris
6. **Face Alignment** — rotasi crop agar mata kiri–kanan sejajar horizontal (landmark-based)
7. **Quality check** — buang crop kosong/noise/hitam
8. **Resize** — `INTER_LANCZOS4` / `INTER_AREA` → 224×224
9. **Video output** — simpan `video_asli_with_bbox.mp4` dengan bbox hijau + 5 titik landmark

In [7]:
def scale_coords_landmarks_np(
    img1_shape: tuple,
    coords: np.ndarray,
    img0_shape: tuple,
    ratio_pad: tuple | None = None,
) -> np.ndarray:
    """
    Peta ulang koordinat landmark (10 kolom) dari ruang letterbox ke resolusi frame asli.
    Logika selaras dengan scale_coords_landmarks di detect_face.py.
    """
    coords = coords.astype(np.float32).copy()
    if coords.ndim == 1:
        coords = coords.reshape(1, -1)

    if ratio_pad is None:
        gain = min(img1_shape[0] / img0_shape[0], img1_shape[1] / img0_shape[1])
        pad = (
            (img1_shape[1] - img0_shape[1] * gain) / 2,
            (img1_shape[0] - img0_shape[0] * gain) / 2,
        )
    else:
        gain = ratio_pad[0][0]
        pad = ratio_pad[1]

    coords[:, [0, 2, 4, 6, 8]] -= pad[0]
    coords[:, [1, 3, 5, 7, 9]] -= pad[1]
    coords[:, :10] /= gain

    coords[:, [0, 2, 4, 6, 8]] = np.clip(coords[:, [0, 2, 4, 6, 8]], 0, img0_shape[1])
    coords[:, [1, 3, 5, 7, 9]] = np.clip(coords[:, [1, 3, 5, 7, 9]], 0, img0_shape[0])
    return coords


def detect_faces_in_frame(
    frame_bgr: np.ndarray,
    model,
    conf_threshold: float = CONF_THRESHOLD,
    imgsz: int = INFERENCE_SIZE,
) -> np.ndarray:
    """
    Deteksi wajah dengan pipeline resmi YOLOv5-Face:
    letterbox → inferensi → NMS face → scale_coords ke resolusi frame asli.

    Menggantikan autoshape() agar gain/pad letterbox selalu konsisten saat cropping.
    """
    repo_path = str(YOLOV5_FACE_REPO.resolve())
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    from utils.datasets import letterbox
    from utils.general import check_img_size, non_max_suppression_face, scale_coords

    img0 = frame_bgr
    stride = int(model.model.stride.max())
    inference_size = check_img_size(imgsz, s=stride)

    img, ratio, pad = letterbox(img0, new_shape=inference_size, auto=False)
    img_input = np.ascontiguousarray(img.transpose(2, 0, 1)[None])
    img_tensor = torch.from_numpy(img_input).to(YOLO_DEVICE).float() / 255.0

    with torch.no_grad():
        pred = model.model(img_tensor)[0]

    iou_thres = getattr(model, 'iou', 0.45)
    classes = getattr(model, 'classes', None)
    pred = non_max_suppression_face(
        pred,
        conf_thres=conf_threshold,
        iou_thres=iou_thres,
        classes=classes,
    )
    det = pred[0]
    if det is None or len(det) == 0:
        return np.empty((0, 16), dtype=np.float32)

    det = det.clone()
    det[:, :4] = scale_coords(
        img.shape[:2],
        det[:, :4],
        img0.shape[:2],
        ratio_pad=(ratio, pad),
    )
    if det.shape[1] >= 15:
        landmarks = scale_coords_landmarks_np(
            img.shape[:2],
            det[:, 5:15].cpu().numpy(),
            img0.shape[:2],
            ratio_pad=(ratio, pad),
        )
        det[:, 5:15] = torch.from_numpy(landmarks).to(det.device, dtype=det.dtype)

    return det.cpu().numpy()


def clamp_bbox_symmetric(
    x1: float,
    y1: float,
    x2: float,
    y2: float,
    frame_w: int,
    frame_h: int,
) -> tuple[int, int, int, int]:
    """
    Clamp bbox ke batas frame dengan pergeseran simetris agar area crop
    tidak 'hilang' hanya di satu sisi (mis. dagu terpotong karena mentok bawah).
    """
    box_w = x2 - x1
    box_h = y2 - y1

    if x1 < 0:
        x2 -= x1
        x1 = 0
    if x2 > frame_w:
        x1 -= x2 - frame_w
        x2 = frame_w
    if y1 < 0:
        y2 -= y1
        y1 = 0
    if y2 > frame_h:
        y1 -= y2 - frame_h
        y2 = frame_h

    x1 = max(0.0, x1)
    y1 = max(0.0, y1)
    x2 = min(float(frame_w), x2)
    y2 = min(float(frame_h), y2)

    # Jika masih terlalu kecil setelah clamp, kembalikan bbox valid terbesar yang muat
    if x2 - x1 < 2:
        cx = (x1 + x2) / 2
        half = min(box_w / 2, frame_w / 2)
        x1, x2 = max(0, cx - half), min(frame_w, cx + half)
    if y2 - y1 < 2:
        cy = (y1 + y2) / 2
        half = min(box_h / 2, frame_h / 2)
        y1, y2 = max(0, cy - half), min(frame_h, cy + half)

    return int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))


def expand_face_crop_bbox(
    bbox: tuple,
    frame_shape: tuple,
    padding_ratio: float = BBOX_PADDING_RATIO,
    square: bool = SQUARE_CROP,
    forehead_bias: float = FOREHEAD_BIAS_RATIO,
) -> tuple[int, int, int, int]:
    """
    Perluas bbox wajah dengan padding proporsional + opsi crop persegi.

    - Padding dihitung dari ukuran bbox deteksi (bukan frame penuh).
    - Crop persegi memakai sisi terpanjang agar alis/bibir tidak terpotong horizontal.
    - Sedikit bias ke atas (forehead_bias) untuk mengamankan area dahi/alis mikro-ekspresi.
    - Koordinat akhir di-clamp simetris ke dalam dimensi video asli.
    """
    frame_h, frame_w = frame_shape[:2]
    x_min, y_min, x_max, y_max = bbox

    box_w = x_max - x_min
    box_h = y_max - y_min
    pad_x = box_w * padding_ratio
    pad_y = box_h * padding_ratio

    x_min -= pad_x
    x_max += pad_x
    y_min -= pad_y
    y_max += pad_y

    if square:
        side = max(x_max - x_min, y_max - y_min)
        cx = (x_min + x_max) / 2.0
        cy = (y_min + y_max) / 2.0 - side * forehead_bias
        x_min = cx - side / 2.0
        x_max = cx + side / 2.0
        y_min = cy - side / 2.0
        y_max = cy + side / 2.0

    return clamp_bbox_symmetric(x_min, y_min, x_max, y_max, frame_w, frame_h)


def draw_face_detection_overlay(
    frame_bgr: np.ndarray,
    bbox: tuple,
    landmarks: np.ndarray | None = None,
    confidence: float | None = None,
) -> np.ndarray:
    """
    Visualisasi deteksi wajah pada frame video: bbox hijau + 5 titik landmark.

    Landmark YOLOv5-Face / WiderFace (urutan resmi):
    0 = mata kiri, 1 = mata kanan, 2 = hidung, 3 = sudut bibir kiri, 4 = sudut bibir kanan
    """
    h, w = frame_bgr.shape[:2]
    line_th = max(2, round(0.002 * (h + w) / 2) + 1)
    x1, y1, x2, y2 = bbox

    cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 255, 0), line_th, cv2.LINE_AA)

    if landmarks is not None and len(landmarks) >= 5:
        # Warna berbeda per titik (selaras detect_face.py) — mudah dibedakan saat demo skripsi
        landmark_colors = [
            (255, 0, 0),    # mata kanan — biru
            (0, 255, 0),    # mata kiri — hijau
            (0, 0, 255),    # hidung — merah
            (0, 255, 255),  # mulut kanan — kuning
            (255, 255, 0),  # mulut kiri — cyan
        ]
        radius = line_th + 2
        for i, (px, py) in enumerate(landmarks[:5]):
            cv2.circle(
                frame_bgr,
                (int(round(px)), int(round(py))),
                radius,
                landmark_colors[i],
                -1,
                lineType=cv2.LINE_AA,
            )

    label = f'Face {confidence:.2f}' if confidence is not None else 'Face Detected'
    cv2.putText(
        frame_bgr,
        label,
        (x1, max(y1 - 10, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 0),
        2,
        cv2.LINE_AA,
    )
    return frame_bgr


def select_best_face_bbox(
    detections: np.ndarray,
    frame_shape: tuple,
    frame_index: int = 0,
    verbose: bool = True,
) -> tuple | None:
    """
    Seleksi bounding box YOLO terbaik dengan center bias.

    Aturan:
    1. Rasio proporsional h/w harus antara 0.8 – 1.8 (menolak bahu/bayangan).
    2. Jika ada >1 kandidat valid, pilih bbox terdekat ke titik tengah frame (center bias).

    Parameters
    ----------
    detections : np.ndarray
        Output YOLOv5-Face shape (N, 16): [x_min, y_min, x_max, y_max, conf, 10 landmarks, cls].
    frame_shape : tuple
        Shape frame (H, W, C).
    frame_index : int
        Indeks frame untuk logging.
    verbose : bool
        Cetak log bbox yang ditolak.

    Returns
    -------
    dict | None
        {'bbox': (x1,y1,x2,y2), 'landmarks': (5,2) array, 'confidence': float} atau None.
    """
    if detections is None or len(detections) == 0:
        return None

    frame_h, frame_w = frame_shape[:2]
    frame_center_x = frame_w / 2.0
    frame_center_y = frame_h / 2.0

    candidates = []

    for det in detections:
        x_min, y_min, x_max, y_max = det[:4].astype(np.float32)
        width = x_max - x_min
        height = y_max - y_min

        # Filter ukuran minimum
        if height < MIN_FACE_HEIGHT_PX or width < MIN_FACE_HEIGHT_PX:
            if verbose:
                print(f'  [REJECT] Frame {frame_index}: bbox terlalu kecil ({width}x{height}px)')
            continue

        # --- Filter 1: Aspect Ratio (h / w) — tolak bbox non-wajah ---
        aspect_ratio = height / max(width, 1)
        if aspect_ratio < FACE_ASPECT_RATIO_MIN or aspect_ratio > FACE_ASPECT_RATIO_MAX:
            if verbose:
                print(
                    f'  [REJECT] Frame {frame_index}: rasio h/w={aspect_ratio:.2f} '
                    f'di luar rentang [{FACE_ASPECT_RATIO_MIN}, {FACE_ASPECT_RATIO_MAX}] '
                    f'(kemungkinan bahu/bayangan)'
                )
            continue

        # --- Filter 2: Center Bias — pakai pusat landmark jika tersedia (lebih akurat dari bbox) ---
        if len(det) >= 15:
            landmarks = det[5:15].reshape(5, 2)
            center_x = float(landmarks[:, 0].mean())
            center_y = float(landmarks[:, 1].mean())
        else:
            center_x = (x_min + x_max) / 2.0
            center_y = (y_min + y_max) / 2.0
        distance = np.sqrt(
            (center_x - frame_center_x) ** 2 + (center_y - frame_center_y) ** 2
        )

        candidates.append({
            'bbox': (
                int(round(x_min)),
                int(round(y_min)),
                int(round(x_max)),
                int(round(y_max)),
            ),
            'distance': distance,
            'confidence': float(det[4]),
            'det': det,
        })

    if not candidates:
        return None

    best = min(candidates, key=lambda c: c['distance'])
    landmarks = None
    if len(best['det']) >= 15:
        landmarks = best['det'][5:15].reshape(5, 2).astype(np.float32)

    return {
        'bbox': best['bbox'],
        'landmarks': landmarks,
        'confidence': best['confidence'],
    }


def landmarks_to_crop_space(
    landmarks: np.ndarray,
    crop_bbox: tuple,
) -> np.ndarray:
    """Pindahkan koordinat landmark dari ruang frame penuh ke ruang crop lokal."""
    x1, y1, _, _ = crop_bbox
    local = landmarks.astype(np.float32).copy()
    local[:, 0] -= x1
    local[:, 1] -= y1
    return local


def align_face_by_landmarks(
    face_bgr: np.ndarray,
    landmarks: np.ndarray,
    left_eye_idx: int = 0,
    right_eye_idx: int = 1,
) -> np.ndarray:
    """
    Face alignment: luruskan wajah dengan memutar crop agar mata kiri & kanan sejajar horizontal.

    Langkah:
    1. Hitung sudut garis antara mata kiri (landmark 0) dan mata kanan (landmark 1).
    2. Rotasi gambar mengitari titik tengah kedua mata.
    3. Perluas kanvas + padding hitam agar tidak ada piksel wajah terpotong setelah rotasi.

    Parameters
    ----------
    face_bgr : np.ndarray
        Crop wajah BGR (H, W, 3).
    landmarks : np.ndarray
        Koordinat landmark shape (5, 2) dalam ruang crop lokal.
    left_eye_idx, right_eye_idx : int
        Indeks mata kiri & kanan pada array landmark.

    Returns
    -------
    np.ndarray
        Crop wajah yang sudah diluruskan (mata horizontal).
    """
    if face_bgr is None or face_bgr.size == 0:
        return face_bgr
    if landmarks is None or len(landmarks) <= max(left_eye_idx, right_eye_idx):
        return face_bgr

    left_eye = landmarks[left_eye_idx].astype(np.float32)
    right_eye = landmarks[right_eye_idx].astype(np.float32)

    d_x = float(right_eye[0] - left_eye[0])
    d_y = float(right_eye[1] - left_eye[1])
    if abs(d_x) < 1e-6 and abs(d_y) < 1e-6:
        return face_bgr

    # Sudut kemiringan mata terhadap horizontal (derajat)
    angle_deg = float(np.degrees(np.arctan2(d_y, d_x)))

    # Pusat rotasi = titik tengah antara kedua mata
    eyes_center = (
        (left_eye[0] + right_eye[0]) / 2.0,
        (left_eye[1] + right_eye[1]) / 2.0,
    )

    h, w = face_bgr.shape[:2]
    rotation_matrix = cv2.getRotationMatrix2D(eyes_center, angle_deg, 1.0)

    # Perluas kanvas agar seluruh gambar muat setelah rotasi (padding hitam di luar)
    cos_a = abs(rotation_matrix[0, 0])
    sin_a = abs(rotation_matrix[0, 1])
    new_w = int(h * sin_a + w * cos_a)
    new_h = int(h * cos_a + w * sin_a)

    rotation_matrix[0, 2] += (new_w - w) / 2.0
    rotation_matrix[1, 2] += (new_h - h) / 2.0

    aligned = cv2.warpAffine(
        face_bgr,
        rotation_matrix,
        (new_w, new_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0),
    )
    return aligned


def is_valid_crop(crop_bgr: np.ndarray) -> bool:
    """
    Memvalidasi kualitas crop sebelum resize.

    Menolak crop yang dominan hitam, terlalu gelap, atau variance terlalu rendah
    (ciri crop kosong / noise digital akibat deteksi gagal).
    """
    if crop_bgr is None or crop_bgr.size == 0:
        return False

    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    mean_val = float(gray.mean())
    std_val = float(gray.std())

    if mean_val < MIN_CROP_MEAN or std_val < MIN_CROP_STD:
        return False

    black_ratio = float((gray < BLACK_PIXEL_THRESHOLD).sum()) / gray.size
    if black_ratio > MAX_BLACK_PIXEL_RATIO:
        return False

    return True


def crop_and_resize(
    frame_bgr: np.ndarray,
    bbox: tuple,
    size: tuple = CROP_SIZE,
    landmarks: np.ndarray | None = None,
    align_face: bool = ENABLE_FACE_ALIGNMENT,
) -> np.ndarray | None:
    """
    Safe cropping → (opsional) face alignment → resize ke ukuran standar input GCN.

    bbox diharapkan sudah dalam koordinat frame asli (setelah scale_coords + expand_face_crop_bbox).
    Jika landmark tersedia dan align_face=True, crop diputar agar mata sejajar horizontal.
    """
    h, w = frame_bgr.shape[:2]
    x_min, y_min, x_max, y_max = bbox

    x_min = max(0, x_min)
    y_min = max(0, y_min)
    x_max = min(w, x_max)
    y_max = min(h, y_max)

    if x_max <= x_min or y_max <= y_min:
        return None

    crop = frame_bgr[y_min:y_max, x_min:x_max].copy()

    if align_face and landmarks is not None and len(landmarks) >= 2:
        local_landmarks = landmarks_to_crop_space(landmarks, bbox)
        crop = align_face_by_landmarks(crop, local_landmarks)

    if not is_valid_crop(crop):
        return None

    target_w, target_h = size
    crop_h, crop_w = crop.shape[:2]

    if crop_h < target_h or crop_w < target_w:
        interpolation = cv2.INTER_LANCZOS4
    else:
        interpolation = cv2.INTER_AREA

    return cv2.resize(crop, size, interpolation=interpolation)


def extract_faces_from_video(
    video_path: str | Path,
    output_dir_for_this_video: str | Path,
    frame_stride: int = FRAME_STRIDE,
    verbose: bool = True,
) -> dict:
    """
    Mengekstraksi ROI wajah dari video input.

    Tahap pre-processing: video mentah → YOLOv5-Face → filter heuristik → crop 224×224
    + video output dengan bounding box (`video_asli_with_bbox.mp4`).

    Parameters
    ----------
    video_path : str | Path
        Path ke berkas video input.
    output_dir_for_this_video : str | Path
        Direktori output crop wajah untuk video ini.
    frame_stride : int
        Interval pengambilan frame (1 = setiap frame).
    verbose : bool
        Tampilkan log peringatan untuk frame yang dibuang.

    Returns
    -------
    dict
        Statistik ekstraksi (saved, skipped, alasan skip, dll.).
    """
    video_path = Path(video_path)
    output_dir = Path(output_dir_for_this_video)
    output_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f'Gagal membuka video: {video_path}')

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 0 or np.isnan(fps):
        fps = 30.0

    # --- Inisialisasi VideoWriter untuk output bbox visualization ---
    bbox_video_path = output_dir / 'video_asli_with_bbox.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(str(bbox_video_path), fourcc, fps, (frame_w, frame_h))

    if not video_writer.isOpened():
        bbox_video_path = output_dir / 'video_asli_with_bbox.avi'
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        video_writer = cv2.VideoWriter(str(bbox_video_path), fourcc, fps, (frame_w, frame_h))

    if not video_writer.isOpened():
        raise RuntimeError('Gagal menginisialisasi VideoWriter untuk output bbox.')

    saved_count = 0
    skipped_no_detection = 0
    skipped_heuristic = 0
    skipped_invalid_crop = 0
    frame_index = 0
    saved_index = 0

    pbar = tqdm(total=total_frames, desc=f'Ekstraksi: {video_path.name}', leave=False)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        pbar.update(1)
        display_frame = frame.copy()

        if frame_index % frame_stride == 0:
            # --- Tahap 1–2: Deteksi wajah + scale_coords ke resolusi frame asli ---
            detections = detect_faces_in_frame(frame, yolo_model, conf_threshold=CONF_THRESHOLD)
            num_raw_detections = len(detections)

            # --- Tahap 3: Filter Heuristik Wajah + Center Bias ---
            face_result = select_best_face_bbox(
                detections, frame.shape, frame_index=frame_index, verbose=verbose
            )

            if face_result is None:
                if num_raw_detections > 0:
                    skipped_heuristic += 1
                else:
                    skipped_no_detection += 1
            else:
                bbox = face_result['bbox']
                landmarks = face_result['landmarks']
                confidence = face_result['confidence']

                # --- Gambar bbox + 5 landmark (mata, hidung, mulut) pada frame asli ---
                draw_face_detection_overlay(
                    display_frame,
                    bbox,
                    landmarks=landmarks,
                    confidence=confidence,
                )

                # --- Tahap 4: Safe crop — padding proporsional + square crop + clamp simetris ---
                crop_bbox = expand_face_crop_bbox(
                    bbox,
                    frame.shape,
                    padding_ratio=BBOX_PADDING_RATIO,
                    square=SQUARE_CROP,
                    forehead_bias=FOREHEAD_BIAS_RATIO,
                )
                face_img = crop_and_resize(
                    frame,
                    crop_bbox,
                    CROP_SIZE,
                    landmarks=landmarks,
                    align_face=ENABLE_FACE_ALIGNMENT,
                )

                if face_img is None:
                    skipped_invalid_crop += 1
                    if verbose:
                        print(f'  [SKIP] Frame {frame_index}: crop tidak valid (kosong/noise)')
                else:
                    save_name = f'frame_{saved_index:04d}.jpg'
                    cv2.imwrite(
                        str(output_dir / save_name),
                        face_img,
                        [cv2.IMWRITE_JPEG_QUALITY, 95],
                    )
                    saved_count += 1
                    saved_index += 1

        # --- Simpan frame ke video output (dengan/tanpa bbox) ---
        video_writer.write(display_frame)
        frame_index += 1

    pbar.close()
    cap.release()
    video_writer.release()

    summary = {
        'video': video_path.name,
        'total_frames_read': frame_index,
        'faces_saved': saved_count,
        'skipped_no_detection': skipped_no_detection,
        'skipped_heuristic': skipped_heuristic,
        'skipped_invalid_crop': skipped_invalid_crop,
        'fps': fps,
        'output_dir': str(output_dir),
        'bbox_video_path': str(bbox_video_path),
        'face_model_used': 'yolov5m-face',
    }
    return summary

## 5. Pipeline Batch Processing — Looping Direktori Video

Seluruh berkas video pada folder `input_videos/` diproses secara batch. Setiap video menghasilkan sub-folder tersendiri di `output_cropped_faces/` yang berisi urutan frame wajah hasil crop.

In [8]:
def collect_video_files(input_dir: Path) -> list[Path]:
    """
    Mengumpulkan daftar path video yang valid dari direktori input.

    Parameters
    ----------
    input_dir : Path
        Direktori berisi video mentah.

    Returns
    -------
    list[Path]
        Daftar path video terurut.
    """
    videos = [
        p for p in sorted(input_dir.iterdir())
        if p.is_file() and p.suffix in VIDEO_EXTENSIONS
    ]
    return videos


def run_batch_face_extraction(input_dir: Path, output_root: Path) -> pd.DataFrame:
    """
    Menjalankan pipeline ekstraksi wajah untuk seluruh video dalam satu direktori.

    Parameters
    ----------
    input_dir : Path
        Folder video input (CASME II / SAMM / custom).
    output_root : Path
        Folder root output crop wajah.

    Returns
    -------
    pd.DataFrame
        Tabel ringkasan hasil ekstraksi per video.
    """
    import pandas as pd

    video_files = collect_video_files(input_dir)

    if not video_files:
        print(f'[PERINGATAN] Tidak ada video di: {input_dir}')
        print('Silakan letakkan file .mp4 / .avi / .mov ke folder tersebut.')
        return pd.DataFrame()

    summaries = []

    for video_path in tqdm(video_files, desc='Batch Processing Video'):
        # Sub-folder output: nama video tanpa ekstensi
        video_output_dir = output_root / video_path.stem

        try:
            summary = extract_faces_from_video(video_path, video_output_dir)
            summaries.append(summary)
            print(f"  ✓ {summary['video']}: {summary['faces_saved']} wajah tersimpan → {summary['output_dir']}")
        except Exception as e:
            print(f"  ✗ Gagal memproses {video_path.name}: {e}")

    return pd.DataFrame(summaries)


# --- Jalankan pipeline batch ---
results_df = run_batch_face_extraction(INPUT_VIDEOS_DIR, OUTPUT_CROPPED_FACES_DIR)
results_df

Batch Processing Video: 100%|██████████| 1/1 [12:20<00:00, 740.91s/it]


  ✓ expression_try.mov: 4108 wajah tersimpan → /Users/stefanieagahari/Micro-Expression-Detector/python test/output_cropped_faces/expression_try


,video,total_frames_read,faces_saved,skipped_no_detection,skipped_heuristic,skipped_invalid_crop,fps,output_dir,bbox_video_path,face_model_used
0,expression_try.mov,4108,4108,0,0,0,29.381333,/Users/stefanieagahari/Micro-Expression-Detect...,/Users/stefanieagahari/Micro-Expression-Detect...,yolov5m-face


## 6. Verifikasi Hasil Ekstraksi (Opsional)

Cell berikut menampilkan sample crop wajah dari video pertama yang berhasil diproses, sebagai bukti visual tahap pre-processing.

In [9]:
import matplotlib.pyplot as plt

if results_df.empty:
    print('Belum ada hasil ekstraksi. Pastikan folder input_videos/ berisi file video.')
else:
    sample_output = Path(results_df.iloc[0]['output_dir'])
    sample_images = sorted(sample_output.glob('frame_*.jpg'))[:6]

    if not sample_images:
        print(f'Tidak ada crop tersimpan di {sample_output}')
    else:
        fig, axes = plt.subplots(2, 3, figsize=(10, 7))
        for ax, img_path in zip(axes.flatten(), sample_images):
            img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(img_path.name, fontsize=8)
            ax.axis('off')
        plt.suptitle(f'Sample ROI Wajah — {sample_output.name}', fontsize=12)
        plt.tight_layout()
        plt.show()

/var/folders/00/s7mtbstd50n0721w72q8t3_80000gn/T/ipykernel_28975/2456682816.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Uji Coba Satu Video (Opsional)

Gunakan cell di bawah untuk menguji **satu file video** secara manual tanpa batch processing.

In [10]:
# Ganti path di bawah dengan video uji Anda
SINGLE_VIDEO = PROJECT_ROOT / 'uploads' / 'expression_try__online-video-cutter.com_.mp4'
SINGLE_OUTPUT = OUTPUT_CROPPED_FACES_DIR / 'single_test'

if SINGLE_VIDEO.exists():
    summary = extract_faces_from_video(SINGLE_VIDEO, SINGLE_OUTPUT)
    print('Ringkasan ekstraksi:')
    for k, v in summary.items():
        print(f'  {k}: {v}')
else:
    print(f'Video tidak ditemukan: {SINGLE_VIDEO}')
    print('Ubah variabel SINGLE_VIDEO ke path video yang valid.')

Video tidak ditemukan: /Users/stefanieagahari/Micro-Expression-Detector/python test/uploads/expression_try__online-video-cutter.com_.mp4
Ubah variabel SINGLE_VIDEO ke path video yang valid.


---

## Kesimpulan Tahap 1

Output notebook ini berupa **urutan gambar wajah tercrop** (`frame_XXXX.jpg`) di folder `output_cropped_faces/<nama_video>/`. Data tersebut siap digunakan pada **Tahap 2: Klasifikasi Spatio-Temporal** menggunakan arsitektur **Graph Convolutional Network (GCN)**.

### Struktur output
```
output_cropped_faces/
├── video_sampel_1/
│   ├── frame_0000.jpg
│   ├── frame_0001.jpg
│   └── ...
└── video_sampel_2/
    └── ...
```

### Langkah selanjutnya (di luar notebook ini)
1. Bangun representasi graf dari urutan frame wajah (node = landmark/frame, edge = relasi temporal/spatial)
2. Latih model GCN / 3D-CNN / ViT untuk klasifikasi mikro ekspresi
3. Evaluasi dengan confusion matrix dan metrik F1-score